Steps to preprocess MODIS data for phenology tracking. Note: GEE runs into memory issues when you try manipulating any of the big data for plotting/viewing/etc. So I am exporting these as geotiffs and reading them in as rasters to python to be able to look at the processing steps.

Author: Tayler LaSharr

Date: September 2024

In [43]:
## first we will read in necessary python libraries and intialize the GEE module
## you will have to authenticate GEE on your machine to use this code

import ee
import matplotlib.pyplot as plt
from datetime import datetime, timezone, timedelta
import geopandas as gpd
from shapely.geometry import box
import pandas as pd
import geemap
import os
import pprint
import time

import json

## if you need to authenticate GEE use this code (will give you a link to allow you to authenticate it online )
# ee.Authenticate()

## initialize the Earth Engine module
ee.Initialize(project='ee-tlasharr')

Define winter period and the date range that we are working with.

In [44]:
## set thresholds and things to manipulate

## define the winter period for your study area
start_month_day = '11-01'
end_month_day = '03-30'

## temporal window to smooth the data
smoothing_window_size = 3

## set a threshold for the snow cover
snow_threshold = 25

## get the time period we care about
start_date_data = '2017-01-01'
end_date_data = '2023-12-31'


Obtain Data.

This step reads in the different data pieces that we care about (modis NDVI, MODIS snow, and mule deer polygons) and define the time intervals that you want to classify winter as anbd the time series for the data you are interested in. This differs very slighly from the R code and uses the day-months instead of the Julian date. This difference is solely to help deal with leap years in systems where winter may end before the end of Feburary when I am working in desert systems.

In [45]:

## load in a shapefile or area of interest, this is an assest that is already loaded into GEE
md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/WY_Sublette') ## "normal" temperate environment
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/UT_Paunsaugunt') ## some snow in winter
# md_cooridor = ee.FeatureCollection('projects/ee-tlasharr/assets/AZ_RainbowValley') ## some snow in winter

## get a bounding box for clipping purposes
bounding_box = md_cooridor.geometry().bounds()

## alternatively, you can create a bounding box from coordinates.
# bounding_box = ee.Geometry.Rectangle([-112.125449, 40.010084, -102.721152, 45.717045])

## load the MODIS NDVI data
modis_ndvi = ee.ImageCollection('MODIS/061/MOD09Q1') \
                .filterDate(start_date_data, end_date_data) \
                .select('sur_refl_b01', 'sur_refl_b02', 'QA', 'State')

## calculate NDVI
def calc_ndvi(image):
    ndvi = image.expression(
        '((NIR - RED) / (NIR + RED))', {
            'NIR': image.select('sur_refl_b02'),
            'RED': image.select('sur_refl_b01')
        }).rename('NDVI')
    return image.addBands(ndvi)

## add the NDVI band to the modis collection
modis_ndvi = modis_ndvi.map(calc_ndvi)

## get the projection of the MODIS NDVI data to reproject the bounding box
modis_projection = modis_ndvi.first().projection()
modis_scale = modis_projection.nominalScale()

## reproject the bounding box to the same projection as MODIS
reprojected_bounding_box = bounding_box.transform(modis_projection, modis_scale)

## clip the modis data to the bounding box
modis_ndvi = modis_ndvi.map(lambda image: image.clip(reprojected_bounding_box))

## get snow cover from MODIS
modis_snow = ee.ImageCollection('MODIS/061/MOD10A1') \
                .filterDate(start_date_data, end_date_data) \
                .select('NDSI_Snow_Cover') \
                .map(lambda image: image.reproject(modis_projection).clip(reprojected_bounding_box))

## make sure the time property is there and correct (i.e., dates for the images)
def check_time_start(image):
    return ee.Algorithms.If(
        image.get('system:time_start'),
        image,
        image.set('system:time_start', ee.Date(image.date()).millis())
    )


modis_ndvi = modis_ndvi.map(lambda img: ee.Image(check_time_start(img)))

## save target projection and scale
target_projection = modis_ndvi.first().projection().getInfo()
target_crs = target_projection['crs']
target_scale = modis_ndvi.first().projection().nominalScale().getInfo()

## just check on a couple of things
print('Projection of MODIS NDVI data:', modis_projection.getInfo())
print('Reprojected Bounding Box:', reprojected_bounding_box.getInfo())
print("Bands available in the first image (NDVI):", modis_ndvi.first().bandNames().getInfo())
print("Bands available in the first image (Snow):", modis_snow.first().bandNames().getInfo())

print("Target projection:", target_projection)
print("Target scale:", target_scale)


print("Number of Images (ndvi):", modis_ndvi.size().getInfo())
print("Number of Images (snow):", modis_snow.size().getInfo())


Projection of MODIS NDVI data: {'type': 'Projection', 'crs': 'SR-ORG:6974', 'transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}
Reprojected Bounding Box: {'geodesic': False, 'crs': {'type': 'name', 'properties': {'name': 'SR-ORG:6974'}}, 'type': 'Polygon', 'coordinates': [[[-9195643.896612, 4639491.792059001], [-9017206.931857, 4639491.792059001], [-8957782.74624, 4686382.625733], [-8897873.318735, 4733273.459407], [-8837481.894627, 4780164.293081], [-8776611.745312, 4827055.126755999], [-8950287.693141999, 4827055.126755999], [-9012362.371173, 4780164.293081], [-9073948.850745, 4733273.459407], [-9135043.795726, 4686382.625733], [-9195643.896612, 4639491.792059001]]]}
Bands available in the first image (NDVI): ['sur_refl_b01', 'sur_refl_b02', 'QA', 'State', 'NDVI']
Bands available in the first image (Snow): ['NDSI_Snow_Cover']
Target projection: {'type': 'Projection', 'crs': 'SR-ORG:6974', 'transform': [231.65635826395825, 0, -20015109.354, 

Get an image collection of the filtered NDVI data and an image collection of the snow cover data.

First function removes low-quality observations with the QA band. This function will create an image collection called 'NDVIFiltered' that has filtered NDVI data during the time period you are interested in (i.e., removes poor quality pixels, snow, and clouds). This is the equivalent of the 'NDVI' rasters produced in step 1b of the R code.

The second function creates a snow mask; this differs from the mask that Jerod and Shannon had made, this will instead create a mask if a NDVI pixel occurs at a point in time where 2 weeks prior and 1 week after the date of the NDVI image has average snow cover that exceeds a set threshold.


In [46]:
## this function removes poor quality pixels, pixels with cloud cover, and pixels with snow
def quality_and_state_mask(image):

    ## get both the state and quality bands
    state = image.select('State')
    quality = image.select('QA')

    ## state band masks
    good_cloud_state = state.bitwiseAnd(3).lt(2)  ## clear or cloudy, exclude mixed and not set
    no_cloud_shadow = state.bitwiseAnd(1 << 2).eq(0)  ## no cloud shadow
    is_land = state.rightShift(3).bitwiseAnd(7).eq(1)  ## land (not in Jerod's code)
    no_snow_ice = state.bitwiseAnd(1 << 12).eq(0)  ## no snow/ice

    ## quality band masks
    good_quality = quality.bitwiseAnd(3).eq(0)  ## highest quality

    ## combine masks: apply all conditions
    combined_mask = good_cloud_state.And(no_cloud_shadow).And(good_quality).And(no_snow_ice).And(is_land)

    ## apply the mask to the NDVI band specifically
    ndvi_masked = image.updateMask(combined_mask).select('NDVI')

    ## then, get rid of outliers
    ndvi_masked =  ndvi_masked.updateMask(ndvi_masked.gte(-1).And(ndvi_masked.lte(2)))

    ## return the masked image
    return ndvi_masked

## this function is to extract snow information and create a mask from the NDSI snow daily cover info
def create_snow_mask_collection(NDVICollection, SnowCollection, snow_threshold=50):
    
    def generate_snow_mask(ndvi_image):
        # Get the date of the current NDVI image
        image_date = ee.Date(ndvi_image.get('system:time_start'))
        
        # Define buffer period (2 weeks before and 1 week after)
        buffer_period_pre = 14  # days before
        buffer_period_post = 7  # days after
        
        # Calculate mean snow cover within the buffer period
        snow_images = SnowCollection.filterDate(
            image_date.advance(-buffer_period_pre, 'day'), 
            image_date.advance(buffer_period_post, 'day')
        ).mean()
        
        # Generate a binary mask (1 for mask if snow cover >= threshold, 0 otherwise)
        snow_mask = snow_images.select('NDSI_Snow_Cover').gte(snow_threshold).rename('Snow_Mask')
        
        # Set system time start to match the original NDVI image for synchronization
        snow_mask = snow_mask.set('system:time_start', ndvi_image.get('system:time_start'))
        
        return snow_mask

    # Map over the NDVI collection to create a corresponding mask collection
    snow_mask_collection = NDVICollection.map(generate_snow_mask)
    
    return snow_mask_collection

## apply the quality mask to the MODIS collection
NDVIFiltered = modis_ndvi.map(lambda img: quality_and_state_mask(img))

## apply the snow extraction function to the MODIS collection
SnowCover = create_snow_mask_collection(NDVIFiltered, modis_snow, snow_threshold=snow_threshold)

## just check on a couple of things
print('Filtered MODIS information:', NDVIFiltered.getInfo())
print("Bands available in the first image:", NDVIFiltered.first().bandNames().getInfo())
print("Number of Images:", NDVIFiltered.size().getInfo())

print('Filtered MODIS Snow information (NDVI):', SnowCover.getInfo())
print("Bands available in the first image (Snow):", SnowCover.first().bandNames().getInfo())
print("Number of Images:", SnowCover.size().getInfo())


Filtered MODIS information: {'type': 'ImageCollection', 'bands': [], 'version': 1730833435912892, 'id': 'MODIS/061/MOD09Q1', 'properties': {'system:is_global': 1}, 'features': [{'type': 'Image', 'bands': [{'id': 'NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [1810, 811], 'origin': [46704, 12762], 'crs': 'SR-ORG:6974', 'crs_transform': [231.65635826395825, 0, -20015109.354, 0, -231.65635826395834, 7783653.637669001]}], 'version': 1649921740560961, 'id': 'MODIS/061/MOD09Q1/2017_01_01', 'properties': {'system:footprint': {'geodesic': False, 'crs': {'type': 'name', 'properties': {'name': 'SR-ORG:6974'}}, 'type': 'Polygon', 'coordinates': [[[-9195643.896612, 4639491.792059001], [-9017206.931857, 4639491.792059001], [-8957782.74624, 4686382.625733], [-8897873.318735, 4733273.459407], [-8837481.894627, 4780164.293081], [-8776611.745312, 4827055.126755999], [-8950287.693141999, 4827055.126755999], [-9012362.371173, 4780164.293081], [-9073948.850745, 4733273.459

In [47]:

# Fetch the first NDVI image and its corresponding snow mask
first_ndvi_image = NDVIFiltered.first()
first_snow_mask_image = SnowCover.first()

# Initialize the map centered on your area of interest
import geemap
Map = geemap.Map(center=[42.5, -107], zoom=6)  # Adjust center coordinates as needed

# Add the NDVI image to the map
Map.addLayer(first_ndvi_image, 
             {'bands': 'NDVI', 'min': 0, 'max': 1, 'palette': ['blue', 'white', 'green']}, 
             'Filtered NDVI (1)')

# Add the snow mask to the map
Map.addLayer(first_snow_mask_image, 
             {'bands': 'Snow_Mask', 'min': 0, 'max': 1, 'palette': ['white', 'black']}, 
             'Snow Mask')

# Add a layer control panel
Map.addLayerControl()

# Display the map
Map


Map(center=[42.5, -107], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(…

Steps 1 - 4. Calculations. Reclassify outliers, reclassify winter values, check if a pixel has been snow-free for a set number of consecutive images, mask pizels that are in winter or covered by snow, apply median filter.

Steps 1 and 2 in this section are the same as the steps in Merkle and Albeke's code for processing NDVI. Step 3 deviates from their methods. Merkle and Albeke methods require a pixel  to be snow free for 3 consecutive time steps in order to retain the NDVI value. This process doesn't work great in the more arid environments, so I shifted that methodology here. Instead, I created a snow mask where if the average snow cover was less than a set threshold for 2 weeks prior and 1 week after the current time step, it would stay as the NDVI value, otherwise it would be masked.

Step 4 is the same as the Merkle and Albeke methods.

In [48]:
## step 1: reclassify outliers
def ndviCALC_step1(NDVICollection):
    
    ## reclassify outliers (mask NDVI values less than 0 and greater than 2)
    def reclassify_outliers(image):
        return image.updateMask(image.gte(0).And(image.lte(2)))
    
    NDVICollection = NDVICollection.map(reclassify_outliers)
    return NDVICollection

## step 2: reclassify values during winter as 2
def ndviCALC_step2(NDVICollection, start_winter_month_day='11-01', end_winter_month_day='03-30'):

    ## split the winter start and end dates into months and days
    start_winter_month, start_winter_day = map(int, start_winter_month_day.split('-'))
    end_winter_month, end_winter_day = map(int, end_winter_month_day.split('-'))

    ## function to classify whether the date is in the winter period
    def classify_winter(image):
        ## extract the date of the image
        date = ee.Date(image.get('system:time_start'))
        year = date.get('year')
        
        ## define winter start and end dates for that year
        winter_start = ee.Date.fromYMD(year, start_winter_month, start_winter_day)
        winter_end = ee.Date.fromYMD(year, end_winter_month, end_winter_day)

        ## use date difference to identify if the image falls within winter
        is_in_winter = date.difference(winter_start, 'day').gte(0).And(date.difference(winter_end, 'day').lte(0))

        ## mask the image during winter (i.e., replace NDVI values with 2 during winter)
        image_winter_adjusted = image.where(is_in_winter, ee.Image.constant(2))

        return image_winter_adjusted

    ## map the function over the collection
    NDVICollection_winter_adjusted = NDVICollection.map(classify_winter)
    return NDVICollection_winter_adjusted

## step 3: mask pixels that are covered by snow
def ndviCALC_step3(NDVICollection, SnowCollection):
    
    ## function to apply the snow mask to each NDVI image
    def apply_snow_mask(ndvi_image):
        
        ## find the corresponding snow mask by matching dates
        snow_mask = SnowCollection.filter(ee.Filter.equals(
            leftField='system:time_start', 
            rightField='system:time_start'
        )).first()
        
        ## if there's no snow mask for this date, create a default mask (no snow)
        snow_mask = ee.Image(ee.Algorithms.If(snow_mask, snow_mask, ee.Image.constant(0).rename('Snow_Mask')))
        
        ## mask NDVI image where snow mask is 1 (indicating snow cover)
        return ndvi_image.updateMask(snow_mask.eq(0))  # retain only where snow_mask is 0

    ## apply the snow mask function across the NDVI collection
    masked_ndvi_collection = NDVICollection.map(apply_snow_mask)
    
    return masked_ndvi_collection

## step 4: apply median filter
def ndviCALC_step4(image_collection, window_size=smoothing_window_size):
    
    image_collection = image_collection.map(lambda img: img.select('NDVI').toFloat()) 

    def smooth_image(current_image):
        all_images = image_collection.sort('system:time_start')
        all_images_list = all_images.toList(all_images.size())

        image_index = all_images_list.indexOf(current_image)
        half_window = ee.Number(window_size // 2)

        ## calculate the start and end indices
        start_index = ee.Number(image_index).subtract(half_window).max(0)
        end_index = ee.Number(image_index).add(half_window).min(all_images.size().subtract(1))

        ## get the window of images
        window_images = all_images_list.slice(start_index, end_index.add(1))
        window_collection = ee.ImageCollection.fromImages(window_images)

        ## compute the median
        median_image = window_collection.median().toFloat()  # Keep NDVI in float

        ## return the median image with the original properties
        return median_image.copyProperties(current_image, current_image.propertyNames())

    ## apply the smoothing function to the collection
    smoothed_collection = image_collection.map(smooth_image)
    return smoothed_collection

## run first 5 steps sequentially
NDVIProcessed1 = ndviCALC_step1(NDVIFiltered)
print("Step 1 complete: Removed outliers from data.")
print("Number of Images:", NDVIFiltered.size().getInfo())

NDVIProcessed2 = ndviCALC_step2(NDVIProcessed1, start_winter_month_day='11-01', end_winter_month_day='03-30')
print("Step 2 complete: Reclassified values during winter.")
print("Number of Images:", NDVIFiltered.size().getInfo())

NDVIProcessed3 = ndviCALC_step3(NDVIProcessed2, SnowCover)
print("Step 3 complete: Apply a mask based on the snow cover image collection.")
print("Number of Images:", NDVIFiltered.size().getInfo())

NDVIProcessed4 = ndviCALC_step4(NDVIProcessed3, window_size=smoothing_window_size)
print("Step 4 complete: Applied moving window with a window size of 3.")
print("Number of Images:", NDVIFiltered.size().getInfo())


Step 1 complete: Removed outliers from data.
Number of Images: 322
Step 2 complete: Reclassified values during winter.
Number of Images: 322
Step 3 complete: Apply a mask based on the snow cover image collection.
Number of Images: 322
Step 4 complete: Applied moving window with a window size of 3.
Number of Images: 322


Steps 5 - 8. Calculate the baseline value, mask NDVI values that are covered with snow, and floor values below the baseline and covered by snow during the winter period.


In [49]:
## step 5: add a winter mask
def ndviCALC_step5(NDVICollection, start_month_day='11-01', end_month_day='03-30'):

    ## parse the start and end month-day strings
    start_month, start_day = map(int, start_month_day.split('-'))
    end_month, end_day = map(int, end_month_day.split('-'))
    
    def add_winter_mask(image):
        
        ## get the date and year of the image
        date = ee.Date(image.get('system:time_start'))
        year = date.get('year')
        
        ## define winter start and end dates for the current year
        winter_start = ee.Date.fromYMD(year, start_month, start_day)
        winter_end = ee.Date.fromYMD(year, end_month, end_day)
        
        ## create a condition to determine if the image is in the winter period
        is_in_winter = date.difference(winter_start, 'day').gte(0).And(
            date.difference(winter_end, 'day').lte(0)
        )
        
        ## add the winter mask as a new band
        return image.addBands(ee.Image(is_in_winter).rename('winter_mask'))
    
    ## apply the winter mask to each image in the collection
    NDVICollection_with_winter_mask = NDVICollection.map(add_winter_mask)
    
    return NDVICollection_with_winter_mask

## step 6: add a snow mask
def ndviCALC_step6(NDVICollection, SnowCover):

    def add_snow_mask(ndvi_image):
        
        ## find the corresponding snow mask by matching dates
        snow_mask = SnowCover.filter(ee.Filter.equals(
            leftField='system:time_start', 
            rightField='system:time_start'
        )).first()
        
        ## if there's no matching snow mask, use a default mask with no snow (0)
        snow_mask = ee.Image(ee.Algorithms.If(snow_mask, snow_mask, ee.Image.constant(0).rename('Snow_Mask')))
        
        ## add the snow mask as a new band to the NDVI image
        return ndvi_image.addBands(snow_mask)
    
    ## apply the snow mask addition to each image in the NDVI collection
    NDVICollection_with_snow_mask = NDVICollection.map(add_snow_mask)
    
    return NDVICollection_with_snow_mask

## step 7: calculate the floor (0.025 quantile)
def ndviCALC_step7(image_collection, snow_collection, snow_threshold, quantile=0.025):

    ## function to mask snow and negative values
    def mask_snow_and_negatives(ndvi_image):
        
        ## get the date of the NDVI image
        image_date = ee.Date(ndvi_image.get('system:time_start'))

        ## create a buffer period around the NDVI image date (e.g., 16 days)
        buffer_period_pre = 14  # 2 weeks before
        buffer_period_post = 3  # 1 week after
        snow_images = snow_collection.filterDate(image_date.advance(-buffer_period_pre, 'day'), image_date.advance(buffer_period_post, 'day')).mean()

        ## create a snow mask where NDSI_Snow_Cover >= snow_threshold
        snow_mask = snow_images.select('NDSI_Snow_Cover').gte(snow_threshold)

        ## apply the snow mask to the NDVI image (exclude values where snow cover exceeds the threshold)
        return ndvi_image.updateMask(snow_mask.Not()).updateMask(ndvi_image.gte(0))

    ## apply the snow mask to each image in the collection
    snow_and_negatives_masked = image_collection.map(mask_snow_and_negatives)

    ## calculate the number of images used to get the baseline
    image_count = snow_and_negatives_masked.select('NDVI').reduce(ee.Reducer.count()).rename('image_count')

    ## calculate the 0.025 quantile (or any specified quantile) for NDVI, excluding snow-covered pixels
    baseline_ndvi = snow_and_negatives_masked.select('NDVI').reduce(ee.Reducer.percentile([quantile * 100])).rename('NDVI_baseline')

    return baseline_ndvi.addBands(image_count)


## step 8: apply corrections to the NDVI collection based on snow cover and winter periods
def ndviCALC_step8(NDVICollection, baseline_ndvi, restrict_to_winter=True):

    def apply_corrections(image):

        ## select masks and NDVI band
        winter_mask = image.select('winter_mask').reduce(ee.Reducer.max()).rename('winter_mask') ## ensure single band
        snow_mask = image.select('Snow_Mask').reduce(ee.Reducer.max()).rename('Snow_Mask') ## ensure single band
        ndvi = image.select('NDVI') ## NDVI values
        
        ## decide whether to restrict corrections to winter months
        if restrict_to_winter:
            correction_condition = winter_mask.And(snow_mask)
        else:
            correction_condition = snow_mask
        
        ## apply corrections, set NDVI to baseline where conditions are met
        corrected_ndvi = ndvi.where(correction_condition.Or(ndvi.lt(baseline_ndvi.select('NDVI_baseline'))), baseline_ndvi.select('NDVI_baseline'))
        corrected_ndvi = corrected_ndvi.unmask(baseline_ndvi.select('NDVI_baseline'))
        
        ## add the corrected NDVI as an updated band
        return image.addBands(corrected_ndvi.rename('NDVI'), overwrite=True)
    
    ## apply corrections to each image in the NDVI collection
    corrected_NDVICollection = NDVICollection.map(apply_corrections)
    
    return corrected_NDVICollection


# Step 5: Add the winter mask to each image
NDVIProcessed5 = ndviCALC_step5(NDVIProcessed4, start_month_day, end_month_day)
print("Step 5 complete: Add winter mask.")
print("Number of Images:", NDVIFiltered.size().getInfo())

# Step 6: Add the snow mask to each image
NDVIProcessed6 = ndviCALC_step6(NDVIProcessed5, SnowCover)
print("Step 6 complete: Add snow mask.")
print("Number of Images:", NDVIFiltered.size().getInfo())

## Step 7: Calculate the baseline floor
baseline_ndvi = ndviCALC_step7(NDVIFiltered, modis_snow, snow_threshold=snow_threshold)
print("Step 7 complete: Calculated Baseline Values.")

# Step 8: Apply corrections based on winter and snow mask
corrected_NDVI = ndviCALC_step8(NDVIProcessed6, baseline_ndvi, restrict_to_winter=True)
print("Step 8 complete: Add baseline values where snow exists.")
print("Number of Images:", NDVIFiltered.size().getInfo())


Step 5 complete: Add winter mask.
Number of Images: 322
Step 6 complete: Add snow mask.
Number of Images: 322
Step 7 complete: Calculated Baseline Values.
Step 8 complete: Add baseline values where snow exists.
Number of Images: 322


In [107]:
def apply_neighbor_baseline_correction(ndvi_image, baseline_image, all_images, target_crs, target_scale):
    
    ## reproject the current NDVI image to the target CRS and scale
    ndvi_image = ndvi_image.reproject(crs=target_crs, scale=target_scale)

    ## reproject all images to the target CRS and scale
    all_images_list = all_images.toList(all_images.size())
    image_index = all_images_list.indexOf(ndvi_image)

    previous_image = ee.Image(ee.Algorithms.If(image_index.gt(0), all_images_list.get(image_index.subtract(1)), None))
    next_image = ee.Image(ee.Algorithms.If(image_index.lt(all_images.size().subtract(1)), all_images_list.get(image_index.add(1)), None))

    ## only reproject if the image is not None
    previous_image = ee.Algorithms.If(previous_image, ee.Image(previous_image).reproject(crs=target_crs, scale=target_scale), None)
    next_image = ee.Algorithms.If(next_image, ee.Image(next_image).reproject(crs=target_crs, scale=target_scale), None)

    def is_baseline(image):
        return image.select('NDVI').eq(baseline_image.select('NDVI_baseline')).reduceRegion(
            reducer=ee.Reducer.allNonZero(),
            geometry=image.geometry(),
            scale=target_scale
        ).values().get(0)

    prev_set_to_baseline = ee.Algorithms.If(previous_image, is_baseline(ee.Image(previous_image)), ee.Number(0))
    next_set_to_baseline = ee.Algorithms.If(next_image, is_baseline(ee.Image(next_image)), ee.Number(0))

    ## determine if both neighboring images are set to the baseline
    both_neighbors_set_to_baseline = ee.Number(prev_set_to_baseline).And(ee.Number(next_set_to_baseline))

    ## define corrections based on neighbors
    def correct_to_baseline():
        return baseline_image.select('NDVI_baseline').rename('NDVI')

    def keep_current_ndvi():
        return ndvi_image.select('NDVI')

    ## apply corrections based on neighbors
    corrected_ndvi = ee.Image(ee.Algorithms.If(both_neighbors_set_to_baseline, correct_to_baseline(), keep_current_ndvi()))
    return ndvi_image.addBands(corrected_ndvi.rename('NDVI'), overwrite=True)


## apply_neighbor_check_to_collection function to run it for the entire image collection
def apply_neighbor_check_to_collection(image_collection, baseline_image, target_crs, target_scale):
    all_images = image_collection.sort('system:time_start')
    
    def check_neighbors(image):
        corrected_image = apply_neighbor_baseline_correction(image, baseline_image, all_images, target_crs, target_scale)
        return corrected_image.select(['NDVI']).rename('NDVI')
    corrected_collection = image_collection.map(check_neighbors)
    return corrected_collection


## set the target projection and scale from your baseline NDVI or the first image in your collection
target_projection = modis_ndvi.first().projection().getInfo()
target_crs = target_projection['crs']
target_scale = modis_ndvi.first().projection().nominalScale().getInfo()

## apply the neighbor correction with consistent projections
corrected_modis_ndvi = apply_neighbor_check_to_collection(corrected_NDVI, baseline_ndvi, target_crs, target_scale)
print("Number of Images:", NDVIFiltered.size().getInfo())


Number of Images: 322


In [ ]:
## Function to generate yearly date ranges
def generate_yearly_date_ranges(start_date, end_date):
    start_year = datetime.strptime(start_date, "%Y-%m-%d").year
    end_year = datetime.strptime(end_date, "%Y-%m-%d").year
    yearly_ranges = []

    for year in range(start_year, end_year + 1):
        start_of_year = f"{year}-01-01"
        end_of_year = f"{year}-12-31"
        yearly_ranges.append((start_of_year, end_of_year))

    return yearly_ranges

## Step 1: Generate Yearly Date Ranges
start_date_data = '2017-01-01'
end_date_data = '2018-12-31'
yearly_ranges = generate_yearly_date_ranges(start_date_data, end_date_data)

## Function to create a grid of bounding boxes
def create_grid(bounding_box, rows, cols):
    bounds = bounding_box.bounds().getInfo()['coordinates'][0]
    min_lon, min_lat = bounds[0]
    max_lon, max_lat = bounds[2]

    lon_step = (max_lon - min_lon) / cols
    lat_step = (max_lat - min_lat) / rows

    grid = []
    for i in range(rows):
        for j in range(cols):
            lon1 = min_lon + lon_step * j
            lat1 = min_lat + lat_step * i
            lon2 = lon1 + lon_step
            lat2 = lat1 + lat_step
            tile = ee.Geometry.Rectangle([lon1, lat1, lon2, lat2])
            grid.append(tile)

    return grid

## Step 2: Create a grid of bounding boxes
rows, cols = 5, 5
grid = create_grid(bounding_box, rows, cols)

## Function to cast all bands to Float32
def cast_to_float32(image):
    bands = image.bandNames()
    casted_bands = bands.map(lambda b: image.select([b]).toFloat())
    return ee.Image.cat(casted_bands).set('system:time_start', image.get('system:time_start'))

## Step 3: Export each tile for each year
def export_yearly_tiles(corrected_NDVI, grid, yearly_ranges, folder_name='NDVI_Exports'):
    for year_index, (start_date, end_date) in enumerate(yearly_ranges):
        yearly_NDVI = corrected_NDVI.filterDate(start_date, end_date)
        yearly_NDVI = yearly_NDVI.map(cast_to_float32)  # Ensure all bands are Float32
        print(f"Exporting data for year: {start_date} to {end_date}")

        for tile_index, tile in enumerate(grid):
            export_image = yearly_NDVI.mosaic().clip(tile)
            export_task = ee.batch.Export.image.toDrive(
                image=export_image,
                description=f'Wyoming_NDVI_{start_date[:4]}_Tile_{tile_index + 1}',
                scale=500,  # Adjust according to your resolution needs
                region=tile,
                fileFormat='GeoTIFF',
                folder=folder_name
            )
            export_task.start()
            print(f"Started export task for Year {start_date[:4]} Tile {tile_index + 1}")

## Perform the export
export_yearly_tiles(corrected_NDVI, grid, yearly_ranges, folder_name='NDVI_Exports')


Exporting data for year: 2017-01-01 to 2017-12-31
Started export task for Year 2017 Tile 1
Started export task for Year 2017 Tile 2
Started export task for Year 2017 Tile 3
Started export task for Year 2017 Tile 4
Started export task for Year 2017 Tile 5
Started export task for Year 2017 Tile 6
Started export task for Year 2017 Tile 7
Started export task for Year 2017 Tile 8
Started export task for Year 2017 Tile 9
Started export task for Year 2017 Tile 10
Started export task for Year 2017 Tile 11
Started export task for Year 2017 Tile 12
Started export task for Year 2017 Tile 13
Started export task for Year 2017 Tile 14
Started export task for Year 2017 Tile 15
Started export task for Year 2017 Tile 16
Started export task for Year 2017 Tile 17
Started export task for Year 2017 Tile 18
Started export task for Year 2017 Tile 19
Started export task for Year 2017 Tile 20
Started export task for Year 2017 Tile 21
Started export task for Year 2017 Tile 22
Started export task for Year 2017 T

Export the data (at each time step so we can plot and look at them)

In [50]:
# Import the geemap library for visualization
import geemap

# Initialize the map centered on your area of interest
Map = geemap.Map(center=[42.5, -107], zoom=6)  # Adjust center coordinates as needed

first_snow_mask_image = SnowCover.first()
baseline_image = baseline_ndvi

# NDVI collection
ndvi_list = NDVIProcessed1.toList(NDVIProcessed1.size())
first_ndvi_image = NDVIProcessed1.first()
twentieth_ndvi_image = ee.Image(ndvi_list.get(19))  # Index 19 for the 20th image

ndvi_list = NDVIProcessed2.toList(NDVIProcessed2.size())
first_ndvi_image2 = NDVIProcessed2.first()
twentieth_ndvi_image2 = ee.Image(ndvi_list.get(19))  # Index 19 for the 20th image

ndvi_list = NDVIProcessed3.toList(NDVIProcessed3.size())
first_ndvi_image3 = NDVIProcessed3.first()
twentieth_ndvi_image3 = ee.Image(ndvi_list.get(19))  # Index 19 for the 20th image

ndvi_list = NDVIProcessed4.toList(NDVIProcessed4.size())
first_ndvi_image4 = NDVIProcessed4.first()
twentieth_ndvi_image4 = ee.Image(ndvi_list.get(19))  # Index 19 for the 20th image

ndvi_list = NDVIProcessed5.toList(NDVIProcessed5.size())
first_ndvi_image5 = NDVIProcessed5.first()
twentieth_ndvi_image5 = ee.Image(ndvi_list.get(19))  # Index 19 for the 20th image

snow_list = SnowCover.toList(SnowCover.size())
snow_image20 = ee.Image(snow_list.get(3))  # Index 19 for the 20th image

ndsi_snow_list = modis_snow.toList(modis_snow.size())
ndsi_snow_image20 = ee.Image(ndsi_snow_list.get(3))  # Index 19 for the 20th image

# Add the NDVI image to the map with a color palette for visualization
Map.addLayer(first_ndvi_image, 
             {'bands': 'NDVI', 'min': 0, 'max': 1, 'palette': ['blue', 'white', 'green']}, 
             'Filtered NDVI (1)')

# Add the 20th NDVI image to the map
# Map.addLayer(twentieth_ndvi_image, 
#              {'bands': 'NDVI', 'min': 0, 'max': 1, 'palette': ['blue', 'white', 'green']}, 
#              'Filtered NDVI (20) - Step 1')

# Map.addLayer(twentieth_ndvi_image2, 
#              {'bands': 'NDVI', 'min': 0, 'max': 1, 'palette': ['blue', 'white', 'green']}, 
#              'Filtered NDVI (20)- Step 2')

# Map.addLayer(twentieth_ndvi_image3, 
#              {'bands': 'NDVI', 'min': 0, 'max': 1, 'palette': ['blue', 'white', 'green']}, 
#              'Filtered NDVI (20) - Step 3')

Map.addLayer(snow_image20, 
             {'bands': 'Snow_Mask', 'min': 0, 'max': 1, 'palette': ['white', 'black']}, 
             'Snow Mask (20)')

# Add the snow mask to the map with a binary color palette
Map.addLayer(first_snow_mask_image, 
             {'bands': 'Snow_Mask', 'min': 0, 'max': 1, 'palette': ['white', 'black']}, 
             'Snow Mask')

Map.addLayer(ndsi_snow_image20, 
             {'bands': 'NDSI_Snow_Cover', 'min': 0, 'max': 1, 'palette': ['white', 'black']}, 
             'Snow')

# Add the baseline 
Map.addLayer(baseline_image, 
             {'bands': 'NDVI_baseline', 'min': 0, 'max': 1, 'palette': ['white', 'red']}, 
             'Baseline')

# Add a layer control panel
Map.addLayerControl()

# Display the map
Map


Map(center=[42.5, -107], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(…